# 01. Team Formation

A solo agent is a function. A *team* is an organization. The moment you have more than one agent collaborating on a goal you need answers to questions a single agent never has to ask: *who is on this team, what role do they play, who is allowed to speak to whom, who do I trust?* `arcteam` is the package that owns those answers.

This notebook is the first stop for `arcteam`. We'll cover how a team is configured, how members are registered with cryptographic identity, how role-based addressing works, and how a minimal team comes together end-to-end. Everything runs in-process against `MemoryBackend` — no filesystem, no network, no API key.

**You will learn:**

- What `TeamConfig` carries and the defaults you usually leave alone
- The `Entity` data model — agents and users, with roles and capabilities
- How `EntityRegistry` registers, looks up, filters, and removes members
- How DIDs from `arctrust` bind a registered entity to a verifiable identity
- How to assemble a minimal team end-to-end with `MemoryBackend` + `AuditLogger`
- How team membership produces tamper-evident audit events
- The lifecycle of a team: formation, operation, dissolution
- Where `arcteam` sits in the four-pillar story (and where it stops)

If you haven't read `arctrust/01-identity-did.ipynb` yet, do that first. We cross-reference it here whenever DIDs come up — this notebook does not re-derive DID basics.

## 1. Setup

The setup cell below is the standard Arc walkthrough preamble — it locates the repo root, adds every package's `src/` (and `tests/` where present) to `sys.path`, and loads any `.env` at the root. Identical across every Arc notebook so you can skim past once you've seen it.

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

Smoke-import the public surface we'll touch in this notebook. If any of these fail, fix the environment before going further — every subsequent cell assumes them.

In [ ]:
import arcteam
from arcteam import (
    AuditLogger,
    Entity,
    EntityRegistry,
    EntityType,
    MemoryBackend,
    TeamConfig,
)

print(f"arcteam version: {arcteam.__version__}")
print(
    "Loaded:",
    [
        AuditLogger.__name__,
        Entity.__name__,
        EntityRegistry.__name__,
        EntityType.__name__,
        MemoryBackend.__name__,
        TeamConfig.__name__,
    ],
)

## 2. Why teams — multi-agent coordination beyond solo agents

Arc's stack splits responsibilities cleanly. A single agent (`arcagent`) is a self-contained nucleus — identity, tool registry, policy pipeline, module bus. The runtime loop (`arcrun`) drives a single agent's reason-act cycle. The LLM call surface (`arcllm`) is provider-agnostic. None of those packages know what to do when *more than one* agent needs to coordinate on a shared goal — and that's deliberate. Stacking concerns onto the agent nucleus is how you get unmaintainable Frankenstein agents.

`arcteam` owns the multi-agent layer. From `packages/arcteam/CLAUDE.md`:

| Concern | arcteam's job | NOT arcteam's job |
|---|---|---|
| Team formation | Create, join, leave, dissolve teams | Individual agent lifecycle |
| Task distribution | Assign, delegate, track work across agents | Execute tasks (that's arcrun) |
| Inter-agent comms | Route messages, signed channels, broadcast | LLM calls (that's arcllm) |
| Roster management | Membership, roles, capability discovery | Agent identity (that's arctrust) |
| Team lifecycle | Formation → Active → Dissolving → Dissolved | Agent config (arcagent) |
| Team telemetry | Team-level audit events | Per-agent telemetry (arcagent) |

The mental model: **`arcteam` orchestrates `ArcAgent` instances** — it does not replace or duplicate agent internals. A team is a *registry* of identities plus the conventions that let those identities address one another, exchange signed messages, and leave a tamper-evident trail.

This notebook covers the first half of that — formation and roster. Channels, messaging, persistence, and consensus get their own notebooks (`02`–`04`).

## 3. `TeamConfig` — what it carries

`TeamConfig` is a small Pydantic model that pins down **where the team's state lives** and **how big a single message can be**. It's deliberately tiny — most teams take the defaults and never look at it again. The defaults align with `arcteam`'s "secure by default" posture (HMAC key from environment, 64 KB body cap to bound memory and audit size).

| Field | Default | Meaning |
|---|---|---|
| `root` | `~/.arc/team` | Filesystem root for team state when using `FileBackend` |
| `hmac_key_env` | `"ARCTEAM_HMAC_KEY"` | Env var the audit logger reads its HMAC key from |
| `max_body_bytes` | `65536` (64 KB) | Hard cap on a single `Message.body` |
| `default_poll_limit` | `10` | Default page size for poll/read APIs |

Note what's *not* there — no team name, no member list, no policy. `TeamConfig` carries deployment-level knobs; team identity and roster live in the registry. This separation is intentional: a process can run many teams against one `TeamConfig`.

In [ ]:
# Defaults out of the box.
default_cfg = TeamConfig()
print("root:                ", default_cfg.root)
print("hmac_key_env:        ", default_cfg.hmac_key_env)
print("max_body_bytes:      ", default_cfg.max_body_bytes)
print("default_poll_limit:  ", default_cfg.default_poll_limit)

You override only what you need. Common overrides: pointing `root` at a project-local directory in tests, or shrinking `max_body_bytes` for a deployment that wants smaller envelopes.

In [ ]:
import tempfile

# Project-local team root for an integration test or sandbox deployment.
sandbox_root = Path(tempfile.mkdtemp(prefix="arc_team_demo_"))
test_cfg = TeamConfig(root=sandbox_root, max_body_bytes=8192, default_poll_limit=25)
print("sandbox root:    ", test_cfg.root)
print("max_body_bytes:  ", test_cfg.max_body_bytes)
print("poll_limit:      ", test_cfg.default_poll_limit)
# Untouched fields keep defaults.
assert test_cfg.hmac_key_env == "ARCTEAM_HMAC_KEY"
print("hmac_key_env:    ", test_cfg.hmac_key_env, "(default preserved)")

`TeamConfig` is a Pydantic v2 model, so you get validation, `.model_dump()`, and `.model_validate()` for free. Round-trip it through JSON the way you'd serialize any other Arc config.

In [ ]:
raw = test_cfg.model_dump(mode="json")
print(raw)

restored = TeamConfig.model_validate(raw)
assert restored.max_body_bytes == 8192
print("Round-trip OK:", restored.max_body_bytes == test_cfg.max_body_bytes)

## 4. `Entity` and `EntityRegistry` — the team roster

A *team* is a collection of `Entity` records held in an `EntityRegistry`. An `Entity` is the smallest unit of membership — a Pydantic model that names *who* is on the team and *what they can do*. The registry handles the CRUD surface plus role-based queries.

### `Entity` shape

| Field | Type | Meaning |
|---|---|---|
| `id` | `str` | URI-form identifier — `agent://name` or `user://name` |
| `name` | `str` | Human-readable label |
| `type` | `EntityType` | `AGENT` or `USER` |
| `roles` | `list[str]` | Tags used for role-based addressing |
| `capabilities` | `list[str]` | Free-form capability tags (`"summarize"`, `"sql"`, …) |
| `created` | `str` | ISO-8601 timestamp; auto-set on register if blank |
| `status` | `str` | Lifecycle status — `active`, `inactive`, etc. |
| `workspace_path` | `str \| None` | SPEC-019 — agent's workspace dir; `None` for users |

Entities are addressed by typed URIs (`agent://`, `user://`). The registry rejects duplicates by ID — you cannot register the same identity twice.

In [ ]:
# Build one of each kind. Roles and capabilities are free-form lists.
planner = Entity(
    id="agent://planner",
    name="Planner",
    type=EntityType.AGENT,
    roles=["coordinator", "decomposer"],
    capabilities=["task.split", "task.assign"],
)
researcher = Entity(
    id="agent://researcher",
    name="Researcher",
    type=EntityType.AGENT,
    roles=["worker"],
    capabilities=["web.search", "doc.summarize"],
)
operator = Entity(
    id="user://josh",
    name="Josh",
    type=EntityType.USER,
    roles=["operator"],
)

for ent in (planner, researcher, operator):
    print(f"{ent.id:<22}  type={ent.type.value:<6}  roles={ent.roles}")

### `EntityRegistry` API

`EntityRegistry` wraps a `StorageBackend` (we'll use `MemoryBackend` here) plus an `AuditLogger`. It exposes a small async surface:

| Method | Purpose |
|---|---|
| `register(entity)` | Insert a new entity; raises if the ID already exists |
| `get(entity_id)` | Read one entity by ID; returns `None` if missing |
| `list_entities(role=None)` | Enumerate all entities, optionally filtered |
| `by_role(role)` | Sugar for `list_entities(role=...)` — used for role-based addressing |
| `update_status(id, status)` | Change lifecycle status; emits `entity.status_changed` |
| `update(entity)` | Replace a record (e.g. backfill a workspace_path); emits `entity.updated` |

Every mutation emits an audit event. We'll inspect the audit stream after we register a few entities.

In [ ]:
# Build a registry on top of MemoryBackend + AuditLogger.
backend = MemoryBackend()
audit = AuditLogger(backend, hmac_key=b"demo-key-do-not-use-in-prod")
await audit.initialize()

registry = EntityRegistry(backend, audit)
print("Registry ready:", type(registry).__name__)
print("Backed by:    ", type(backend).__name__)

Register all three entities. `register()` is a write-through that:

1. Rejects duplicates (raises `ValueError`).
2. Auto-sets `created` if blank.
3. Persists the record to the backend.
4. Emits an `entity.registered` audit event.

In [ ]:
await registry.register(planner)
await registry.register(researcher)
await registry.register(operator)

# Read back to confirm everything stuck.
back = await registry.get("agent://planner")
assert back is not None
print("Re-read planner:")
print("  id:       ", back.id)
print("  name:     ", back.name)
print("  roles:    ", back.roles)
print("  created:  ", back.created)
print("  status:   ", back.status)

**Duplicate IDs are rejected.** This is a registry invariant: two members cannot share an ID.

In [ ]:
# this raises — re-registering the same ID is a hard error
try:
    await registry.register(planner)
except ValueError as e:
    print("ValueError:", e)

### Listing and filtering

`list_entities()` returns every entity. `by_role()` filters down to a single role tag. The role check is a simple membership test against `entity.roles`, so an entity with multiple roles surfaces in multiple role queries.

In [ ]:
everyone = await registry.list_entities()
print(f"All entities ({len(everyone)}):")
for e in everyone:
    print(f"  {e.id:<22}  roles={e.roles}")

print()
workers = await registry.by_role("worker")
print(f"Workers ({len(workers)}):", [e.id for e in workers])

operators = await registry.by_role("operator")
print(f"Operators ({len(operators)}):", [e.id for e in operators])

# A role nobody has -> empty list, not an error.
ghosts = await registry.by_role("phantom")
print("Phantom role (empty):", ghosts)

### Status transitions and updates

A team's roster isn't static — members go offline, swap workspaces, change roles. `update_status()` handles the lifecycle bit; `update()` is the general escape hatch for replacing a full record. Both emit dedicated audit events.

In [ ]:
# Take the researcher offline.
await registry.update_status("agent://researcher", "inactive")
researcher_now = await registry.get("agent://researcher")
assert researcher_now is not None
print("researcher.status:", researcher_now.status)

# Backfill a workspace_path via update() (SPEC-019 path).
researcher_now.workspace_path = str(sandbox_root / "agents" / "researcher")
await registry.update(researcher_now)
readback = await registry.get("agent://researcher")
assert readback is not None
print("researcher.workspace_path:", readback.workspace_path)
print("roles preserved:           ", readback.roles)

**Removal — what `arcteam` does and doesn't expose.** The current registry surface intentionally has no `remove()` or `delete()` method on `EntityRegistry`. To take a member out of rotation you mark the status (`inactive`, `dissolved`, etc.) — keeping the record in place preserves the audit trail. The underlying `StorageBackend.delete` exists for low-level callers, but day-to-day team operations use `update_status` instead. This is by design: federal-tier audit requirements (NIST 800-53 AU-9) want immutable history.

In [ ]:
# Soft-removal pattern: status transition rather than physical delete.
await registry.update_status("agent://researcher", "dissolved")
final = await registry.get("agent://researcher")
assert final is not None
print(f"researcher final status: {final.status}")
# Record still exists in the registry — audit chain is intact.
still_listed = await registry.list_entities()
print(f"Total entities after soft-remove: {len(still_listed)}")

### Every mutation leaves an audit footprint

Each `register`, `update_status`, and `update` emits an HMAC-chained audit record into `audit/audit`. The chain is verifiable end-to-end with `AuditLogger.verify_chain()` — same primitive `arctrust.audit` uses, just wrapped for team-scoped events.

In [ ]:
# Read the audit stream directly off the backend.
records = await backend.read_stream("audit", "audit", after_seq=0)
print(f"Audit records: {len(records)}")
for r in records:
    print(f"  seq={r['audit_seq']}  event={r['event_type']:<24}  actor={r['actor_id']}")

# Verify the HMAC chain from end to end.
ok, last = await audit.verify_chain()
print()
print(f"chain valid? {ok}  (verified through seq={last})")
assert ok

## 5. Member identity — DID-bound entries

The `Entity.id` field is a *URI handle* (`agent://planner`). It tells the registry where to file the record, but on its own it doesn't *prove* anything. In Arc, real identity is cryptographic — every agent has a DID backed by an Ed25519 keypair, generated and managed by `arctrust`. (See `arctrust/01-identity-did.ipynb` for the DID model itself; we don't re-derive it here.)

So how do registry handles and DIDs fit together?

**Registry handle vs. cryptographic identity.** The `agent://planner` URI is a *team-local nickname* — convenient for messaging, role queries, and human readability. The DID is the *globally-unique cryptographic identity*. A real team binds the two:

```
agent://planner   →   did:arc:acme:executor/9b43ee77
        (handle)              (verifiable identity)
```

The most common pattern is to stuff the DID and capability metadata onto the `Entity.capabilities` list (or a richer side-store). The registry's job is membership; `arctrust` is the source of truth for what each member can prove. Below we wire both together to show the shape.

In [ ]:
# Build a DID for the planner via arctrust. Same as in arctrust/01.
from arctrust import AgentIdentity

planner_identity = AgentIdentity.generate(org="acme", agent_type="executor")
print("planner DID:        ", planner_identity.did)
print("can sign?           ", planner_identity.can_sign)

Bind the DID to the registry record. We use `capabilities` as a typed bag for facts about the agent — `did:` prefix makes intent obvious to a reader.

In [ ]:
# Add the DID to the registry record. update() writes through and emits audit.
planner_now = await registry.get("agent://planner")
assert planner_now is not None
planner_now.capabilities = [
    *planner_now.capabilities,
    f"did:{planner_identity.did}",
]
await registry.update(planner_now)

readback = await registry.get("agent://planner")
assert readback is not None
print("planner capabilities:")
for cap in readback.capabilities:
    print(f"  - {cap}")

With that binding, any caller holding the registry handle (`agent://planner`) can look up the DID and verify a signature. This is how `arcteam` enables signed inter-agent messaging without baking signature logic into the registry itself: the registry stores the binding; `arctrust` provides verification.

In [ ]:
# Demonstrate the verification path: caller has the handle, looks up the DID,
# verifies a signature with arctrust primitives.
from arctrust import sign as raw_sign, verify as raw_verify

# Planner signs a payload (e.g., a task assignment).
assert planner_identity._signing_key is not None
payload = b"assign:task-001:agent://researcher"
sig = raw_sign(payload, planner_identity._signing_key.encode())

# Verifier path: pull the registry record, extract the DID, verify with arctrust.
record = await registry.get("agent://planner")
assert record is not None
did_caps = [c for c in record.capabilities if c.startswith("did:")]
assert did_caps, "expected DID binding on planner"
print("resolved DID:", did_caps[0].removeprefix("did:"))

# Verify using the cached public key (in production you'd resolve via a trust store).
ok = raw_verify(payload, sig, planner_identity.public_key)
print("signature valid:", ok)
assert ok

**Where the line is.** `arcteam` does not re-implement signing, parent/child derivation, or the policy pipeline. Those primitives live in `arctrust` and are reused as-is — the registry is purely the place where team-scoped *handles* are kept. Notebooks 02 (channels) and 03 (messaging) show how signed messages travel between members; this notebook only sets up the membership graph.

## 6. Forming a team end-to-end — minimal example

Pulling everything together: a fresh team comes up in three steps:

1. **Choose a config and a backend.** `TeamConfig` for deployment knobs, `MemoryBackend` (tests) or `FileBackend` (production) for storage.
2. **Stand up the audit logger.** `AuditLogger` with an HMAC key — chains every team mutation.
3. **Register members.** `EntityRegistry.register(...)` for each `Entity`.

Below is the smallest end-to-end formation. We wrap it in a function so the same code can be reused later in tests.

In [ ]:
async def form_team(
    name: str, members: list[Entity]
) -> tuple[EntityRegistry, AuditLogger, MemoryBackend]:
    """Stand up a fresh in-process team and register members."""
    backend = MemoryBackend()
    audit = AuditLogger(backend, hmac_key=f"{name}-hmac-key".encode())
    await audit.initialize()
    registry = EntityRegistry(backend, audit)
    for member in members:
        await registry.register(member)
    return registry, audit, backend

Use the helper to form a small "research squad" team.

In [ ]:
squad = [
    Entity(
        id="agent://lead",
        name="Lead",
        type=EntityType.AGENT,
        roles=["coordinator"],
        capabilities=["task.assign"],
    ),
    Entity(
        id="agent://scout",
        name="Scout",
        type=EntityType.AGENT,
        roles=["worker", "research"],
        capabilities=["web.search"],
    ),
    Entity(
        id="agent://writer",
        name="Writer",
        type=EntityType.AGENT,
        roles=["worker", "writing"],
        capabilities=["doc.draft"],
    ),
    Entity(id="user://owner", name="Owner", type=EntityType.USER, roles=["operator"]),
]

squad_registry, squad_audit, squad_backend = await form_team("research-squad", squad)
roster = await squad_registry.list_entities()
print(f"research-squad has {len(roster)} members:")
for m in roster:
    print(f"  {m.id:<24}  type={m.type.value:<6}  roles={m.roles}")

Sanity-check the audit chain from formation. Every member registered = one `entity.registered` event = one HMAC-chained record.

In [ ]:
formation_records = await squad_backend.read_stream("audit", "audit", after_seq=0)
print(f"Audit records during formation: {len(formation_records)}")
for r in formation_records:
    print(f"  seq={r['audit_seq']}  {r['event_type']}  actor={r['actor_id']}")
ok, last = await squad_audit.verify_chain()
print(f"chain valid? {ok}  (through seq={last})")
assert ok and last == len(squad)

**Role-based addressing in action.** Once members are registered with `roles`, you can address subsets of the team by role rather than by individual ID. This is the foundation `arcteam`'s messaging layer (notebook `03`) builds `role://` URIs on top of.

In [ ]:
workers = await squad_registry.by_role("worker")
print("Workers:")
for w in workers:
    print(f"  {w.id}  capabilities={w.capabilities}")

coordinators = await squad_registry.by_role("coordinator")
print("Coordinators:", [c.id for c in coordinators])

# Capability-based filtering is *not* a registry primitive — list and filter in code.
researchers = [e for e in roster if "research" in e.roles]
print("With 'research' role:", [r.id for r in researchers])

## 7. Member roles and metadata — what's representable

`Entity.roles` and `Entity.capabilities` are both `list[str]` — string-tag bags. That's a deliberate "boring" choice. Arc deployments differ wildly in their org structures (federal vs. enterprise vs. personal), so `arcteam` deliberately doesn't bake in a fixed role taxonomy. You choose.

That said, *conventions* matter — here's the convention that emerges across Arc components:

| Tag form | Example | Used for |
|---|---|---|
| **Functional roles** | `worker`, `coordinator`, `reviewer` | Role-based addressing (`role://worker`) |
| **Domain roles** | `research`, `writing`, `qa` | Subset filtering, capability discovery |
| **Operator roles** | `operator`, `owner`, `admin` | Authority gates (combine with policy in arctrust) |
| **Capability tags** | `web.search`, `doc.draft`, `sql` | Tool-level capability advertisement |
| **Identity bindings** | `did:arc:acme:executor/...` | DID lookup (see §5) |

Pick a convention and stick to it across a deployment. The registry won't enforce one, but downstream addressing and policy lookups will assume it.

In [ ]:
# Showcase the difference between role and capability tags.
multi_role_agent = Entity(
    id="agent://swiss-army",
    name="Swiss Army",
    type=EntityType.AGENT,
    roles=["worker", "research", "writing", "reviewer"],
    capabilities=["web.search", "doc.draft", "doc.review", "sql"],
)
await squad_registry.register(multi_role_agent)

# Same entity surfaces in multiple role queries.
for role in ("worker", "research", "writing", "reviewer"):
    matches = await squad_registry.by_role(role)
    ids = [m.id for m in matches]
    print(f"role '{role}':  {ids}")

Capabilities aren't a registry-level filter. They're free-form tags that other layers (the agent's tool registry, a discovery service) interpret.

In [ ]:
# Capability discovery — done in caller code, not by the registry.
def find_capable(entities: list[Entity], capability: str) -> list[Entity]:
    return [e for e in entities if capability in e.capabilities]


all_members = await squad_registry.list_entities()
sql_capable = find_capable(all_members, "sql")
print("SQL-capable agents:", [e.id for e in sql_capable])

draft_capable = find_capable(all_members, "doc.draft")
print("Drafting-capable agents:", [e.id for e in draft_capable])

**`workspace_path` (SPEC-019).** Agents have a workspace directory on disk — populated either at registration time (`arc team register --workspace ...`) or backfilled with `arc team backfill-workspaces` for older records. Users have no workspace, so `workspace_path = None`. This is enforced by convention; the registry will accept any value.

In [ ]:
# Show the workspace_path field — agent has one, user does not.
ws_records = [(e.id, e.type.value, e.workspace_path) for e in all_members]
for eid, etype, wp in ws_records:
    print(f"  {eid:<24}  type={etype:<6}  workspace={wp!r}")

## 8. Lifecycle — formation → operation → dissolution

Teams aren't immortal. They're formed for a purpose, operate, and dissolve when the purpose is complete (or the SCIF audit window closes). `arcteam` tracks this lifecycle through `Entity.status` transitions and the audit chain — there's no separate "team object" with a state machine; the team's lifecycle *is* the sequence of registry mutations.

| Phase | What happens | Audit signal |
|---|---|---|
| **Formation** | Members registered; roles assigned | One `entity.registered` per member |
| **Operation** | Status changes, role updates, workspace backfills | `entity.status_changed`, `entity.updated` |
| **Dissolution** | Members marked `dissolved` (soft-remove); audit sealed | Trailing `entity.status_changed` events |

Below we walk a tiny team through all three phases and inspect the resulting audit timeline.

In [ ]:
# --- Phase 1: Formation
sprint = [
    Entity(id="agent://owner", name="Owner", type=EntityType.AGENT, roles=["coordinator"]),
    Entity(id="agent://hand-1", name="Hand-1", type=EntityType.AGENT, roles=["worker"]),
    Entity(id="agent://hand-2", name="Hand-2", type=EntityType.AGENT, roles=["worker"]),
]
sprint_reg, sprint_audit, sprint_backend = await form_team("sprint", sprint)

formation_count = len(await sprint_backend.read_stream("audit", "audit", after_seq=0))
print(f"after formation: {formation_count} audit records")

In [ ]:
# --- Phase 2: Operation
# A worker briefly goes offline, comes back, then both workers backfill workspace paths.
await sprint_reg.update_status("agent://hand-1", "inactive")
await sprint_reg.update_status("agent://hand-1", "active")

sprint_workspaces = sandbox_root / "sprint-workspaces"
for worker_id in ("agent://hand-1", "agent://hand-2"):
    rec = await sprint_reg.get(worker_id)
    assert rec is not None
    rec.workspace_path = str(sprint_workspaces / worker_id.split("://")[-1])
    await sprint_reg.update(rec)

operation_count = len(await sprint_backend.read_stream("audit", "audit", after_seq=0))
print(f"after operation: {operation_count} audit records")

In [ ]:
# --- Phase 3: Dissolution
# Mark everyone dissolved. Records remain for audit; no physical deletes.
for member in await sprint_reg.list_entities():
    await sprint_reg.update_status(member.id, "dissolved")

dissolution_records = await sprint_backend.read_stream("audit", "audit", after_seq=0)
print(f"after dissolution: {len(dissolution_records)} audit records total")
print()
print("Final timeline (last 8 events):")
for r in dissolution_records[-8:]:
    print(f"  seq={r['audit_seq']:>2}  {r['event_type']:<24}  {r['actor_id']}")

# Audit chain still verifies after every transition.
ok, last = await sprint_audit.verify_chain()
print()
print(f"audit chain valid? {ok}  (verified through seq={last})")
assert ok

A few practical notes about lifecycle:

- **Soft-remove > hard delete.** The `dissolved` status keeps the record visible in `list_entities()`. Filtering it out at the application layer is fine; rewriting history isn't.
- **The audit chain is your timeline.** `read_stream("audit", "audit", ...)` is the canonical replay surface. Notebook `04-team-persistence` covers durable backends and recovery.
- **Cross-team isolation.** Each team can run on its own `MemoryBackend` / `FileBackend`. There's no global registry — that's by design (no singleton bottlenecks; teams are independent).

In [ ]:
# Cross-team isolation: squad_registry and sprint_reg never see each other.
squad_ids = {e.id for e in await squad_registry.list_entities()}
sprint_ids = {e.id for e in await sprint_reg.list_entities()}
print("squad ids:  ", sorted(squad_ids))
print("sprint ids: ", sorted(sprint_ids))
print("intersection:", squad_ids & sprint_ids)
assert not (squad_ids & sprint_ids), "teams must be isolated"

### Cleanup

Wipe the temp directory we created for `TeamConfig.root` so this notebook is leave-no-trace.

In [ ]:
import shutil

if sandbox_root.exists():
    shutil.rmtree(sandbox_root)
    print("Removed:", sandbox_root)

## 9. Summary

**What you saw in this notebook:**

- Why `arcteam` exists — the multi-agent layer that orchestrates `ArcAgent` instances without duplicating agent internals.
- `TeamConfig` carries deployment knobs (`root`, `hmac_key_env`, `max_body_bytes`, `default_poll_limit`) and nothing else; team identity and roster live elsewhere.
- `Entity` is the unit of membership: typed URI (`agent://`/`user://`), name, type, roles, capabilities, status, optional workspace path.
- `EntityRegistry` registers, looks up, lists, filters by role, transitions status, and updates records — every mutation emits a chained audit event.
- DIDs from `arctrust` bind the registry handle to a verifiable cryptographic identity. `arcteam` keeps the binding; `arctrust` does the cryptography.
- A team comes up in three steps: pick `TeamConfig` + backend, stand up `AuditLogger`, register members.
- Roles and capabilities are deliberate string-tag bags — convention over schema.
- Lifecycle is *implicit* in the audit chain: formation → operation → dissolution is just a sequence of registry mutations, soft-removed via `update_status("dissolved")` rather than physical deletes.
- Cross-team isolation is a property of the backend, not a registry feature — no singleton, no shared state.

**Public API surface covered:**

- `arcteam.TeamConfig` — defaults, overrides, Pydantic round-trip
- `arcteam.Entity` and `arcteam.EntityType` (`AGENT` / `USER`)
- `arcteam.EntityRegistry` — `register`, `get`, `list_entities`, `by_role`, `update_status`, `update`
- `arcteam.MemoryBackend` (used as `StorageBackend`)
- `arcteam.AuditLogger` — initialize, `verify_chain`
- Cross-references: `arctrust.AgentIdentity`, `arctrust.sign`, `arctrust.verify` for DID-bound members

**Where this fits in the four pillars:**

- **Identity** — every team member maps to a DID via `arctrust` (covered in `arctrust/01-identity-did`).
- **Sign** — signed inter-agent messages use `arctrust.sign` / `arctrust.verify`; `arcteam` stores the binding only.
- **Authorize** — team-scoped policy decisions still go through `arctrust.policy` (`PolicyPipeline`); `arcteam` doesn't ship its own policy engine.
- **Audit** — every registry mutation lands in the HMAC-chained audit stream. Notebook `04-team-persistence` covers durability and recovery.

**Next:**

- `arcteam/02-task-distribution.ipynb` — `Channel`, `MsgType`, `Priority`, and how tasks fan out across the roster.
- `arcteam/03-messaging-channels.ipynb` — `MessagingService` and the `Message` envelope; signed inter-agent communication.
- `arcteam/04-team-persistence.ipynb` — `TeamMemoryService`, durable backends (`FileBackend`), team recovery after a crash.